# Reverse-QTY series C MD, part 1 of 4: prepare the CHARMM-GUI inputs

Taner Karagol and Alper Karagol.

**Runtime > Run all, then close it.** No GPU needed and nothing to fill in. It takes
under a minute.

It fetches the deposited assembly and the UniProt entry, derives the converted
positions instead of reading a list, writes the two builds you will upload to
CHARMM-GUI, and writes the settings checklist beside them. Part 2 picks up from the
CHARMM-GUI output.

## The question

The reverse-QTY rule converts every glutamine, threonine and tyrosine inside a helix,
and it carries no criterion of burial or helical face, so it cannot exclude a dimer
interface. 14-3-3 is the first target of this line whose fold is an obligate dimer.
The interface was measured first, on four structures across three paralogues, and the
answer was the same each time: three of the converted positions lie in it, carrying
19 to 25 per cent of the area buried on dimer formation. Two of the three are the
glutamines of the first helix, and the third is the tyrosine whose hydroxyl hydrogen
bonds across the interface to an arginine and an aspartate of the partner subunit, a
bond the tyrosine to phenylalanine substitution deletes by construction.

These runs test whether that matters. The kill criterion for the series is stated
before anything is built: the converted dimer has to hold its interface in water over
100 ns, measured against a native dimer run in the same conditions, and the series
stops if it does not.

## Why this structure

5IQP is the ligand-free 14-3-3 theta dimer, Xiao et al., Nature 376:188 (1995),
doi 10.1038/376188a0. Its only heteroatoms are sulfate and water, the biological
dimer is the pair in the asymmetric unit so no symmetry has to be applied, and both
chains cover the complete canonical chain of P27348. YWHAQ is also the only one of
the seven paralogues whose converted hydropathicity crosses zero, which is the
behaviour the published application reports for apolipoprotein E.

2BQ0 with P31946 is the structural control and the second build, and this notebook
takes it through the same checks by changing two of the fields below.

## Where it reads and writes

The project folder lives on Drive under the root set below, so the notebook
and the record stay together. Inputs and
the CHARMM-GUI output go under `md/` inside it. Upload the project folder before
running: this notebook refuses to create it, and if the name is wrong it says which
nearby folders it found, because there are unrelated `14-3-3` folders on that Drive
from the isoform line. The project carries its own inputs: the `dssp` source reads
`data/iface_1433.json`, written by `code/iface_1433.py` in this folder, and the
sequences come from `data/paralogue_uniprot.json`, written by
`code/00_fetch_sequences.py`. Nothing here reads another project's files.

## What it refuses to do

A gap longer than one residue, a gap whose flanks sit inside the interface at the
5 angstrom contact cutoff, a converted position with no observed side chain inside
the interface, a disulfide where the checklist says there are none, a chain count
other than two, and any residue whose identity disagrees with the UniProt sequence.
Each of those would put a modelled coordinate inside the measurement.


In [ ]:
#@title Run all starts here: mount Drive, build the two inputs
PROJECT_DIR   = "" #@param {type:"string"}
#@markdown The project folder, used when it is not where DRIVE_ROOT and
#@markdown PROJECT_FOLDER below point. Set it to wherever you put this folder,
#@markdown on Drive or on the Colab disk, and every other path follows from it.
#@markdown Left empty it falls back to those two.
PDB_ID        = "5IQP"   #@param {type:"string"}
#@markdown 5IQP is the ligand-free 14-3-3 theta dimer, sulfate and water only, with
#@markdown both chains covering the complete canonical chain. 2BQ0 is the beta control.
PARALOGUE     = "YWHAQ"  #@param ["YWHAQ", "YWHAB"]
ACCESSION     = "P27348" #@param {type:"string"}
#@markdown P27348 is YWHAQ, 14-3-3 theta. P31946 is YWHAB, 14-3-3 beta, with 2BQ0.
HELIX_SOURCE  = "uniprot" #@param ["uniprot", "dssp"]
#@markdown Which source selects the converted positions. The published Methods used
#@markdown DSSP on a predicted model; this line states its choice instead of implying
#@markdown one. uniprot reads the deposited helix annotation. dssp reads the list
#@markdown iface_1433.py wrote, which must be on Drive as inputs/iface_1433.json.
MAX_GAP       = 1        #@param {type:"integer"}
#@markdown Longest run of unobserved residues CHARMM-GUI may model. A gap is allowed
#@markdown only when the gap and its flanks lie outside the dimer interface, taken
#@markdown at the same 5 angstrom heavy-atom contact cutoff the screen's interface
#@markdown definition uses, so the criterion is not one chosen here.
CONTACT_CUTOFF = 5.0     #@param {type:"number"}
REF_T         = 303.15   #@param {type:"number"}
#@markdown Thermostat temperature for both CHARMM-GUI builds. 303.15 K is the value
#@markdown the APOE reverse-QTY paper ran at, so the two series are comparable, and
#@markdown it is also CHARMM-GUI's own default. It goes into the checklist and into
#@markdown the build spec, and part 2 refuses a build whose mdp disagrees with it.
DRIVE_ROOT    = "/content/drive/MyDrive"  #@param {type:"string"}
#@markdown The folder on Drive that holds the project. Case matters on Colab.
PROJECT_FOLDER = "14-3-3-rQTY" #@param {type:"string"}
#@markdown The project folder inside it. Upload it first: this
#@markdown notebook refuses to create it, so a mistyped name fails instead of
#@markdown scattering files into a new folder.

from google.colab import drive
drive.mount('/content/drive')

import collections, csv, json, math, os, sys, urllib.request

import numpy as np

PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
DATA = os.path.join(PROJECT, "data")
OUT_DIR = os.path.join(PROJECT, "md", "inputs")
RQTY = {"Q": "L", "T": "V", "Y": "F"}
THREE = {"Q": "GLN", "T": "THR", "Y": "TYR", "L": "LEU", "V": "VAL", "F": "PHE"}
AA3 = {"A": "ALA", "R": "ARG", "N": "ASN", "D": "ASP", "C": "CYS", "Q": "GLN",
       "E": "GLU", "G": "GLY", "H": "HIS", "I": "ILE", "L": "LEU", "K": "LYS",
       "M": "MET", "F": "PHE", "P": "PRO", "S": "SER", "T": "THR", "W": "TRP",
       "Y": "TYR", "V": "VAL"}
SIDECHAIN = {
    "GLN": ["CB", "CG", "CD", "OE1", "NE2"],
    "LEU": ["CB", "CG", "CD1", "CD2"],
    "THR": ["CB", "OG1", "CG2"],
    "VAL": ["CB", "CG1", "CG2"],
    "TYR": ["CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ", "OH"],
    "PHE": ["CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ"],
}
BACKBONE = ["N", "CA", "C", "O"]
FULL = {"ALA": ["CB"], "ARG": ["CB", "CG", "CD", "NE", "CZ", "NH1", "NH2"],
        "ASN": ["CB", "CG", "OD1", "ND2"], "ASP": ["CB", "CG", "OD1", "OD2"],
        "CYS": ["CB", "SG"], "GLN": ["CB", "CG", "CD", "OE1", "NE2"],
        "GLU": ["CB", "CG", "CD", "OE1", "OE2"], "GLY": [],
        "HIS": ["CB", "CG", "ND1", "CD2", "CE1", "NE2"],
        "ILE": ["CB", "CG1", "CG2", "CD1"], "LEU": ["CB", "CG", "CD1", "CD2"],
        "LYS": ["CB", "CG", "CD", "CE", "NZ"], "MET": ["CB", "CG", "SD", "CE"],
        "PHE": ["CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ"], "PRO": ["CB", "CG", "CD"],
        "SER": ["CB", "OG"], "THR": ["CB", "OG1", "CG2"],
        "TRP": ["CB", "CG", "CD1", "CD2", "NE1", "CE2", "CE3", "CZ2", "CZ3", "CH2"],
        "TYR": ["CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ", "OH"],
        "VAL": ["CB", "CG1", "CG2"]}


def fatal(msg):
    raise SystemExit("STOP: " + msg)


def check_project():
    """The project folder has to be on Drive already, and be the right one."""
    if not os.path.isdir(DRIVE_ROOT):
        fatal("%s is not there. Mount Drive and check the path; on Colab the "
              "capitalisation has to match." % DRIVE_ROOT)
    if not os.path.isdir(PROJECT):
        near = sorted(d for d in os.listdir(DRIVE_ROOT)
                      if os.path.isdir(os.path.join(DRIVE_ROOT, d))
                      and d.lower().replace(" ", "")[:5]
                      == PROJECT_FOLDER.lower().replace(" ", "")[:5])
        fatal("%s is not on Drive. Upload the project folder "
              "first. Folders in %s whose names start the same way: %s"
              % (PROJECT, DRIVE_ROOT, near or "none"))
    want = ["README.md", os.path.join("data", "paralogue_uniprot.json"), "codes"]
    missing = [w for w in want if not os.path.exists(os.path.join(PROJECT, w))]
    if missing:
        fatal("%s does not carry %s, so it is probably not the project folder this "
              "notebook belongs to. Upload the whole folder, not only this notebook."
              % (PROJECT, ", ".join(missing)))
    print("project   ", PROJECT)
    print("inputs    ", OUT_DIR)


check_project()


def fetch(url, local, what):
    os.makedirs(OUT_DIR, exist_ok=True)
    if os.path.isfile(local):
        print("using the copy already in", OUT_DIR)
        return open(local, errors="replace").read()
    print("downloading", url)
    try:
        txt = urllib.request.urlopen(url, timeout=120).read().decode("utf-8", "replace")
    except Exception as e:
        fatal("could not download the %s (%s). Put it at %s by hand and rerun."
              % (what, e, local))
    open(local, "w").write(txt)
    return txt


# ---------------------------------------------------------------- the sequence
up = json.loads(fetch("https://rest.uniprot.org/uniprotkb/%s?format=json"
                      "&fields=accession,sequence,ft_helix" % ACCESSION,
                      os.path.join(OUT_DIR, "%s.json" % ACCESSION), "UniProt entry"))
SEQ = up["sequence"]["value"]
HELIX = sorted((f["location"]["start"]["value"], f["location"]["end"]["value"])
               for f in up.get("features", []) if f["type"] == "Helix")
if not HELIX:
    fatal("%s carries no helix annotation, so the uniprot source cannot select "
          "positions" % ACCESSION)
print("%s %s, %d residues, %d annotated helices"
      % (PARALOGUE, ACCESSION, len(SEQ), len(HELIX)))

# ------------------------------------------------------------- the coordinates
raw = fetch("https://files.rcsb.org/download/%s.pdb1" % PDB_ID,
            os.path.join(OUT_DIR, "%s.pdb1" % PDB_ID), "deposited assembly")
if raw.count("MODEL ") > 1:
    fatal("%s.pdb1 carries %d models, so the biological dimer is built by symmetry "
          "and this notebook would merge two copies of one subunit"
          % (PDB_ID, raw.count("MODEL ")))
lines, het = [], collections.Counter()
for L in raw.splitlines():
    if L[:6] == "HETATM" and L[17:20] != "HOH":
        het[L[17:20].strip()] += 1
    if not L.startswith("ATOM"):
        continue
    if L[16] not in (" ", "A") or L[26] != " ":
        continue
    lines.append(L)
if not lines:
    fatal("%s.pdb1 carries no usable ATOM records" % PDB_ID)
print("non-water heteroatoms dropped:", dict(het) or "none")

chains = sorted({L[21] for L in lines})
if len(chains) != 2:
    fatal("expected a dimer, found %d chains (%s). The interface this tests is "
          "between two subunits." % (len(chains), ",".join(chains)))

RES = collections.OrderedDict()
for L in lines:
    RES.setdefault((L[21], int(L[22:26])), L[17:20].strip())
XYZ = collections.defaultdict(list)
NAMES = collections.defaultdict(set)
for L in lines:
    XYZ[(L[21], int(L[22:26]))].append((float(L[30:38]), float(L[38:46]),
                                        float(L[46:54])))
    NAMES[(L[21], int(L[22:26]))].add(L[12:16].strip())


def _min_distances():
    """Shortest heavy-atom distance from every residue to the other subunit."""
    out = {}
    for c in chains:
        other = [x for x in chains if x != c][0]
        keys = [k for k in XYZ if k[0] == c]
        far = np.array([p for k in XYZ if k[0] == other for p in XYZ[k]])
        for k in keys:
            here = np.array(XYZ[k])
            d = np.sqrt(((here[:, None, :] - far[None, :, :]) ** 2).sum(-1))
            out[k] = float(d.min())
    return out


MIND = _min_distances()


def clearance(key, other_chain=None):
    return MIND[key]


gaps_allowed = []
for c in chains:
    other = [x for x in chains if x != c][0]
    rs = sorted(r for ch, r in RES if ch == c)
    bad = [(r, SEQ[r - 1], RES[(c, r)]) for r in rs
           if not (1 <= r <= len(SEQ)) or RES[(c, r)] != AA3[SEQ[r - 1]]]
    if bad:
        fatal("chain %s disagrees with %s at %s, so the numbering is not UniProt "
              "numbering" % (c, ACCESSION, bad[:5]))
    runs = [(rs[i] + 1, rs[i + 1] - 1) for i in range(len(rs) - 1)
            if rs[i + 1] - rs[i] > 1]
    print("chain %s: %d residues %d..%d, %d unobserved stretch(es) inside"
          % (c, len(rs), min(rs), max(rs), len(runs)))
    for a, b in runs:
        n = b - a + 1
        d = round(min(clearance((c, a - 1), other), clearance((c, b + 1), other)), 2)
        print("   %s%d..%s%d, %d residue(s), flanks %.2f A from the partner subunit"
              % (SEQ[a - 1], a, SEQ[b - 1], b, n, d))
        if n > MAX_GAP:
            fatal("chain %s is missing %d residues at %d..%d. CHARMM-GUI would model "
                  "that loop; decide before running." % (c, n, a, b))
        if d <= CONTACT_CUTOFF:
            fatal("chain %s is missing %d..%d and its flanks sit %.1f A from the "
                  "partner subunit, which is inside the interface at the %.1f A "
                  "contact cutoff. A modelled loop there would be part of the "
                  "answer." % (c, a, b, d, CONTACT_CUTOFF))
        gaps_allowed.append(dict(chain=c, first=a, last=b, residues=n,
                                 clearance_A=d))

sg = {}
for L in lines:
    if L[17:20].strip() == "CYS" and L[12:16].strip() == "SG":
        sg[(L[21], int(L[22:26]))] = (float(L[30:38]), float(L[38:46]),
                                      float(L[46:54]))
ss = [(a, b) for i, a in enumerate(sorted(sg)) for b in sorted(sg)[i + 1:]
      if math.dist(sg[a], sg[b]) < 2.5]
print("cysteines: %d, disulfides under 2.5 A: %d" % (len(sg), len(ss)))
if ss:
    fatal("found %d disulfide(s) %s. The checklist this notebook writes says there "
          "are none, so it would be wrong." % (len(ss), ss))

PARTIAL = sorted(k for k, v in RES.items()
                 if [a for a in FULL[v] if a not in NAMES[k]])
IFACE = sorted(k for k in RES if MIND[k] <= CONTACT_CUTOFF)
print("\ninterface at the %.1f A contact cutoff: %d residues over %d chains"
      % (CONTACT_CUTOFF, len(IFACE), len(chains)))
if PARTIAL:
    print("side chains not fully observed, %d residue(s), %d of them in the interface:"
          % (len(PARTIAL), len([k for k in PARTIAL if k in set(IFACE)])))
    for k in PARTIAL:
        miss = [a for a in FULL[RES[k]] if a not in NAMES[k]]
        print("   chain %s %s%-4d missing %-18s %5.1f A from the partner subunit%s"
              % (k[0], RES[k], k[1], ",".join(miss), MIND[k],
                 "   INTERFACE" if k in set(IFACE) else ""))

# ------------------------------------------------------- the converted positions
if HELIX_SOURCE == "uniprot":
    POS = [p for p, ch in enumerate(SEQ, 1)
           if ch in RQTY and any(a <= p <= b for a, b in HELIX)]
    SOURCE_NOTE = ("every Q, T and Y inside an annotated helix of %s, %d helices"
                   % (ACCESSION, len(HELIX)))
else:
    path = os.path.join(DATA, "iface_1433.json")
    if not os.path.isfile(path):
        fatal("HELIX_SOURCE is dssp but %s is not there. It is written by "
              "code/iface_1433.py in this project, so rerun that and upload the "
              "project folder again." % path)
    ifj = json.load(open(path))
    hit = [r for r in ifj["structures"] if r["pdb"] == PDB_ID]
    if not hit:
        fatal("iface_1433.json carries no entry for %s" % PDB_ID)
    POS = sorted(hit[0]["conv_dssp"])
    SOURCE_NOTE = ("every Q, T and Y whose DSSP assignment on %s chain A is H, G or I"
                   % PDB_ID)
if not POS:
    fatal("no convertible positions derived")
alt = [p for p, ch in enumerate(SEQ, 1)
       if ch in RQTY and any(a <= p <= b for a, b in HELIX)]
print("\nconverted positions (%s): %d" % (HELIX_SOURCE, len(POS)))
print("   ", ", ".join("%s%d%s" % (SEQ[p - 1], p, RQTY[SEQ[p - 1]]) for p in POS))
if HELIX_SOURCE != "uniprot":
    print("   the annotation source would give %d, differing at %s"
          % (len(alt), [p for p in POS if p not in alt] or "nothing"))
for c in chains:
    missing = [p for p in POS if (c, p) not in RES]
    if missing:
        fatal("chain %s does not observe converted position(s) %s, so the "
              "substitution cannot be applied there" % (c, missing))
    wrong = [(p, RES[(c, p)]) for p in POS if RES[(c, p)] != THREE[SEQ[p - 1]]]
    if wrong:
        fatal("chain %s carries %s where the sequence says otherwise" % (c, wrong[:5]))

MODELLED = [k for k in PARTIAL if k[1] in POS]
for k in MODELLED:
    print("   converted position %s%d in chain %s has no observed side chain, "
          "%.1f A from the partner subunit" % (RES[k], k[1], k[0], MIND[k]))
inside = [k for k in MODELLED if MIND[k] <= CONTACT_CUTOFF]
if inside:
    fatal("converted position(s) %s sit inside the interface at the %.1f A cutoff "
          "and carry no observed side chain, so the substitution there would be "
          "applied to a model and not to the deposited geometry"
          % (inside, CONTACT_CUTOFF))
CONV_IFACE = sorted(p for p in POS
                    if any(MIND[(c, p)] <= CONTACT_CUTOFF for c in chains))
print("   converted positions inside the interface: %d, %s"
      % (len(CONV_IFACE), ", ".join("%s%d" % (SEQ[p - 1], p) for p in CONV_IFACE)))

# ------------------------------------------------------------------- the builds
def keep_set(old3, new3):
    old, new = SIDECHAIN[old3], SIDECHAIN[new3]
    shared = 0
    for x, y in zip(old, new):
        if x != y:
            break
        shared += 1
    return set(BACKBONE) | set(old[:shared]), old[shared:], new[shared:]


def write_build(tag, convert):
    out, touched, rebuilt = [], set(), collections.Counter()
    for L in lines:
        r = int(L[22:26])
        if convert and r in POS:
            old3 = THREE[SEQ[r - 1]]
            new3 = THREE[RQTY[SEQ[r - 1]]]
            keep, dropped, added = keep_set(old3, new3)
            if L[12:16].strip() not in keep:
                continue
            L = L[:17] + "%-3s" % new3 + L[20:]
            touched.add((L[21], r))
            rebuilt["%s to %s" % (old3, new3)] = len(added)
        out.append(L)
    nmut = len(touched)
    path = os.path.join(OUT_DIR, "%s_dimer_%s.pdb" % (PDB_ID, tag))
    with open(path, "w") as fh:
        fh.write("REMARK  reverse-QTY series C MD input, %s, %s %s\n"
                 % (tag, PARALOGUE, ACCESSION))
        fh.write("REMARK  %s chains %s, protein only\n" % (PDB_ID, ",".join(chains)))
        if convert:
            fh.write("REMARK  %d positions converted per the %s source\n"
                     % (len(POS), HELIX_SOURCE))
        fh.write("\n".join(out) + "\nEND\n")
    print("  %-26s %6d atoms%s" % (os.path.basename(path), len(out),
                                   "" if not convert else
                                   ", %d positions converted over %d chains"
                                   % (nmut, len(chains))))
    if convert:
        for k in sorted(rebuilt):
            print("       %-14s CHARMM-GUI rebuilds %d side-chain atom(s)"
                  % (k, rebuilt[k]))
        if nmut != len(POS) * len(chains):
            fatal("converted %d positions, expected %d over %d chains"
                  % (nmut, len(POS) * len(chains), len(chains)))
    return path, len(out)


print("\nbuilds:")
BUILDS = collections.OrderedDict()
BUILDS["native"] = write_build("native", False)
BUILDS["converted"] = write_build("converted", True)

with open(os.path.join(OUT_DIR, "substitutions_%s.csv" % PARALOGUE), "w",
          newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["position", "native", "rQTY", "helix_range", "source"])
    for p in POS:
        hr = next(("%d-%d" % (a, b) for a, b in HELIX if a <= p <= b), "none")
        w.writerow([p, SEQ[p - 1], RQTY[SEQ[p - 1]], hr, HELIX_SOURCE])

spec = dict(pdb_id=PDB_ID, paralogue=PARALOGUE, accession=ACCESSION,
            helix_source=HELIX_SOURCE, source_note=SOURCE_NOTE,
            chains=chains, positions=POS,
            expect={str(p): THREE[SEQ[p - 1]] for p in POS},
            converted_to={str(p): THREE[RQTY[SEQ[p - 1]]] for p in POS},
            modelled_gaps=gaps_allowed, disulfides=len(ss),
            contact_cutoff=CONTACT_CUTOFF,
            interface=["%s %s%d" % (k[0], RES[k], k[1]) for k in IFACE],
            converted_in_interface=CONV_IFACE,
            partial_sidechains=["%s %s%d (%.1f A)" % (k[0], RES[k], k[1], MIND[k])
                                for k in PARTIAL],
            modelled_converted=["%s %s%d" % (k[0], RES[k], k[1]) for k in MODELLED],
            atoms={k: v[1] for k, v in BUILDS.items()},
            ref_t_K=REF_T,
            builds=["native", "converted"])
json.dump(spec, open(os.path.join(OUT_DIR, "build_spec.json"), "w"), indent=2)

CHECKLIST = """CHARMM-GUI settings for the reverse-QTY series C dimer test
==========================================================
Input Generator > Solution Builder > Upload PDB. Run it twice, once per file.

  %s
  %s

Chains            keep both, %s. The interface between them is what this measures.
Disulfides        none. %s has %d cysteines across the two chains and no pair is
                  within 2.5 angstrom, so nothing is to be ticked here. If
                  CHARMM-GUI offers one, something is wrong with the upload.
Missing residues  %s
Terminal groups   standard charged termini, NTER and CTER. The chain ends here are the
                  real ends of the protein, not a cut through the middle of one.
Protonation       defaults, standard protonation at pH 7. The two builds must differ
                  in nothing but the %d substitutions.
Water box         rectangular, 12 angstrom edge distance.
Ions              KCl 0.15 M, neutralising. The two builds carry the same net charge,
                  since every substitution here is neutral to neutral, so the ion
                  counts should match; if they do not, check the upload.
Force field       CHARMM36m.
Dynamics          GROMACS. Temperature %s K, the value the APOE reverse-QTY paper
                  ran at, so the two series are comparable. It is also CHARMM-GUI's
                  own default, so this field can be left as it is.

Unpack each charmm-gui-*.tgz and put the gromacs/ folder on Drive as

  %s/md/native/gromacs
  %s/md/converted/gromacs

Then open the two part 2 notebooks and press Runtime > Run all on each. They are
independent and can run at the same time.

The question these two runs answer: the conversion changes %d positions of this
subunit, of which the ones inside the dimer interface are the risk, so the native and
the converted dimer are run in water under the same conditions and the interface is
compared. Nothing downstream is built until that comparison returns.
""" % (os.path.basename(BUILDS["native"][0]), os.path.basename(BUILDS["converted"][0]),
       ",".join(chains), PDB_ID, len(sg),
       ("none to model." if not gaps_allowed else
        "; ".join("chain %s is missing %s%d%s, %.2f angstrom from the partner subunit, "
                  "so let CHARMM-GUI model it"
                  % (g["chain"], SEQ[g["first"] - 1], g["first"],
                     "" if g["residues"] == 1 else " to %s%d" % (SEQ[g["last"] - 1], g["last"]),
                     g["clearance_A"]) for g in gaps_allowed)),
       len(POS), REF_T, PROJECT, PROJECT, len(POS))

open(os.path.join(OUT_DIR, "CHARMM-GUI-checklist.txt"), "w").write(CHECKLIST)
print("\n" + "=" * 72)
print(CHECKLIST)
print("everything written to", OUT_DIR)


## Done

The two PDBs, `substitutions_YWHAQ.csv`, `build_spec.json` and
`CHARMM-GUI-checklist.txt` are in `md/inputs/` inside the project folder on Drive.
The checklist printed above is the same file, so you can close this notebook and work
from Drive.

Unpack each CHARMM-GUI result into `md/native/gromacs` and `md/converted/gromacs`
beside those inputs. Part 2 reads `build_spec.json` and refuses a folder whose residue
at any converted position does not match what this notebook wrote, so uploading the
two builds in the wrong order is caught before anything runs.
